### Multimodal RAG PDF With Text and Images


In [4]:
!python -m pip install --upgrade pip

Python(67659) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


In [5]:
!pip install pymupdf langchain langchain-community transformers pillow torch scikit-learn faiss-cpu langchain-ollama

Python(67660) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


In [6]:
import fitz  # PyMuPDF
from langchain_core.documents import Document
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import torch
import numpy as np
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage
from sklearn.metrics.pairwise import cosine_similarity
import os
import base64
import io
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_ollama import ChatOllama



In [7]:
###Clip Model
import os

clip_model=CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor=CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()


Python(67663) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Loading weights: 100%|██████████| 398/398 [00:00<00:00, 1590.90it/s, Materializing param=visual_projection.weight]                                
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1e-05,

In [8]:
## Process PDF
pdf_path="/Users/urmitmahida34/projects/Multimodal_RAG_PDF_Text_Images/visual_language_model_few_shot_learning.pdf"
doc=fitz.open(pdf_path)
# Storage for all documents and embeddings
all_docs = []
all_embeddings = []
image_data_store = {}  # Store actual image data for llava

# Text splitter
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)


In [9]:
doc

Document('/Users/urmitmahida34/projects/Multimodal_RAG_PDF_Text_Images/visual_language_model_few_shot_learning.pdf')

In [10]:
def embed_image(image_data):
    """Embed image using CLIP"""
    if isinstance(image_data, str):
        image = Image.open(image_data).convert("RGB")
    else:
        image = image_data
    
    inputs = clip_processor(images=image, return_tensors="pt")
    with torch.no_grad():
        outputs = clip_model.get_image_features(**inputs)
        
        # FIX: If it's an object, get the underlying tensor. 
        # If it's already a tensor, this line won't break it.
        features = outputs.pooler_output if hasattr(outputs, 'pooler_output') else outputs
        
        # Normalize the tensor
        features = torch.nn.functional.normalize(features, p=2, dim=-1)
        
        return features.squeeze().cpu().numpy()

def embed_text(text):
    """Embed text using CLIP."""
    inputs = clip_processor(
        text=[text], # Wrap in list to avoid character-wise tokenization
        return_tensors="pt", 
        padding=True,
        truncation=True,
        max_length=77
    )
    with torch.no_grad():
        outputs = clip_model.get_text_features(**inputs)
        
        # FIX: Ensure we extract the tensor from the output object
        features = outputs.pooler_output if hasattr(outputs, 'pooler_output') else outputs
        
        # Normalize the tensor
        features = torch.nn.functional.normalize(features, p=2, dim=-1)
        
        return features.squeeze().cpu().numpy()

In [11]:
for i,page in enumerate(doc):
    ## process text
    text=page.get_text()
    if text.strip():
        ##create temporary document for splitting
        temp_doc = Document(page_content=text, metadata={"page": i, "type": "text"})
        text_chunks = splitter.split_documents([temp_doc])

        #Embed each chunk using CLIP
        for chunk in text_chunks:
            embedding = embed_text(chunk.page_content)
            all_embeddings.append(embedding)
            all_docs.append(chunk)



    ## process images
    ##Three Important Actions:

    ##Convert PDF image to PIL format
    ##Store as base64 for GPT-4V (which needs base64 images)
    ##Create CLIP embedding for retrieval

    for img_index, img in enumerate(page.get_images(full=True)):
        try:
            xref = img[0]
            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]
            
            # Convert to PIL Image
            pil_image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
            
            # Create unique identifier
            image_id = f"page_{i}_img_{img_index}"
            
            # Store image as base64 for later use with GPT-4V
            buffered = io.BytesIO()
            pil_image.save(buffered, format="PNG")
            img_base64 = base64.b64encode(buffered.getvalue()).decode()
            image_data_store[image_id] = img_base64
            
            # Embed image using CLIP
            embedding = embed_image(pil_image)
            all_embeddings.append(embedding)
            
            # Create document for image
            image_doc = Document(
                page_content=f"[Image: {image_id}]",
                metadata={"page": i, "type": "image", "image_id": image_id}
            )
            all_docs.append(image_doc)
            
        except Exception as e:
            print(f"Error processing image {img_index} on page {i}: {e}")
            continue

doc.close()


In [12]:
all_docs

[Document(metadata={'page': 0, 'type': 'text'}, page_content='Flamingo: a Visual Language Model\nfor Few-Shot Learning\nJean-Baptiste Alayrac*,‡\nJeff Donahue*\nPauline Luc*\nAntoine Miech*\nIain Barr†\nYana Hasson†\nKarel Lenc†\nArthur Mensch†\nKatie Millican†\nMalcolm Reynolds†\nRoman Ring†\nEliza Rutherford†\nSerkan Cabi\nTengda Han\nZhitao Gong\nSina Samangooei\nMarianne Monteiro\nJacob Menick\nSebastian Borgeaud\nAndrew Brock\nAida Nematzadeh\nSahand Sharifzadeh\nMikolaj Binkowski\nRicardo Barreira\nOriol Vinyals\nAndrew Zisserman\nKaren Simonyan*,‡'),
 Document(metadata={'page': 0, 'type': 'text'}, page_content='Mikolaj Binkowski\nRicardo Barreira\nOriol Vinyals\nAndrew Zisserman\nKaren Simonyan*,‡\n* Equal contributions, ordered alphabetically, † Equal contributions, ordered alphabetically,\n‡ Equal senior contributions\nDeepMind\nAbstract\nBuilding models that can be rapidly adapted to novel tasks using only a handful of\nannotated examples is an open challenge for multimodal m

In [13]:
# Create unified FAISS vector store with CLIP embeddings
embeddings_array = np.array(all_embeddings)
embeddings_array

array([[-0.0187145 ,  0.01629316,  0.01300664, ..., -0.10063366,
         0.027182  , -0.0372769 ],
       [ 0.03484819, -0.01359458,  0.00930944, ...,  0.05066207,
         0.0281936 , -0.01315186],
       [ 0.00465729,  0.00185516,  0.01766196, ..., -0.0083179 ,
         0.03666432, -0.05486517],
       ...,
       [-0.0001615 , -0.02516828,  0.01789003, ..., -0.11636708,
         0.01430553, -0.05698925],
       [ 0.02082038, -0.01137797, -0.00871693, ..., -0.04510185,
         0.00418822, -0.03756266],
       [-0.01408871, -0.01143881,  0.00182964, ..., -0.09083608,
         0.0114811 , -0.03807385]], shape=(628, 512), dtype=float32)

In [14]:
all_docs,embeddings_array

([Document(metadata={'page': 0, 'type': 'text'}, page_content='Flamingo: a Visual Language Model\nfor Few-Shot Learning\nJean-Baptiste Alayrac*,‡\nJeff Donahue*\nPauline Luc*\nAntoine Miech*\nIain Barr†\nYana Hasson†\nKarel Lenc†\nArthur Mensch†\nKatie Millican†\nMalcolm Reynolds†\nRoman Ring†\nEliza Rutherford†\nSerkan Cabi\nTengda Han\nZhitao Gong\nSina Samangooei\nMarianne Monteiro\nJacob Menick\nSebastian Borgeaud\nAndrew Brock\nAida Nematzadeh\nSahand Sharifzadeh\nMikolaj Binkowski\nRicardo Barreira\nOriol Vinyals\nAndrew Zisserman\nKaren Simonyan*,‡'),
  Document(metadata={'page': 0, 'type': 'text'}, page_content='Mikolaj Binkowski\nRicardo Barreira\nOriol Vinyals\nAndrew Zisserman\nKaren Simonyan*,‡\n* Equal contributions, ordered alphabetically, † Equal contributions, ordered alphabetically,\n‡ Equal senior contributions\nDeepMind\nAbstract\nBuilding models that can be rapidly adapted to novel tasks using only a handful of\nannotated examples is an open challenge for multimodal

In [15]:
# Create custom FAISS index since we have precomputed embeddings
vector_store = FAISS.from_embeddings(
    text_embeddings=[(doc.page_content, emb) for doc, emb in zip(all_docs, embeddings_array)],
    embedding=None,  # We're using precomputed embeddings
    metadatas=[doc.metadata for doc in all_docs]
)
vector_store

`embedding_function` is expected to be an Embeddings object, support for passing in a function will soon be removed.


In [16]:
# Initialize Ollama llava model
llm = ChatOllama(model="llava", temperature=0)
llm

ChatOllama(model='llava', temperature=0.0)

In [17]:
def retrieve_multimodal(query, k=5):
    """Unified retrieval using CLIP embeddings for both text and images."""
    # Embed query using CLIP
    query_embedding = embed_text(query)
    
    # Search in unified vector store
    results = vector_store.similarity_search_by_vector(
        embedding=query_embedding,
        k=k
    )
    
    return results

In [23]:
## modified create_multimodal_message function to handle both text and image results
def create_multimodal_message(query, retrieved_docs):
    """Create a message with both text and images for Ollama/Llava."""
    content = []
    
    # 1. Add the query
    content.append({
        "type": "text",
        "text": f"Question: {query}\n\n"
    })
    
    # 2. Add text context (if any)
    text_docs = [doc for doc in retrieved_docs if doc.metadata.get("type") == "text"]
    if text_docs:
        text_context = "\n".join([f"[Page {d.metadata['page']}]: {d.page_content}" for d in text_docs])
        content.append({
            "type": "text",
            "text": f"Relevant Text Context:\n{text_context}\n"
        })
    
    # 3. Add images
    image_docs = [doc for doc in retrieved_docs if doc.metadata.get("type") == "image"]
    for doc in image_docs:
        image_id = doc.metadata.get("image_id")
        if image_id in image_data_store:
            # Modern LangChain format for Ollama
            content.append({
                "type": "image_url",
                "image_url": f"data:image/png;base64,{image_data_store[image_id]}"
            })
            
    return HumanMessage(content=content)

In [24]:
## modified multimodal_pdf_rag_pipeline function to use the new message format
from langchain_ollama import ChatOllama

# 1. Initialize the local vision model
# Ensure you have run 'ollama pull llava' in your terminal first
llm = ChatOllama(model="llava", temperature=0)

def multimodal_pdf_rag_pipeline(query):
    """Main pipeline for Local Multimodal RAG."""
    # Retrieve relevant documents (CLIP + FAISS)
    context_docs = retrieve_multimodal(query, k=3) # 'k' reduced for local speed
    
    # Create multimodal message
    message = create_multimodal_message(query, context_docs)
    
    # Invoke Ollama
    # Note: Ollama might take 10-30 seconds depending on your hardware
    response = llm.invoke([message])
    
    # (Optional) Clean up print logic for debugging
    print(f"\n--- [Local Retrieval: Found {len(context_docs)} chunks] ---")
    
    return response.content

In [21]:
def multimodal_pdf_rag_pipeline(query):
    """Main pipeline for multimodal RAG."""
    # Retrieve relevant documents
    context_docs = retrieve_multimodal(query, k=5)
    
    # Create multimodal message
    message = create_multimodal_message(query, context_docs)
    
    # Get response from GPT-4V
    response = llm.invoke([message])
    
    # Print retrieved context info
    print(f"\nRetrieved {len(context_docs)} documents:")
    for doc in context_docs:
        doc_type = doc.metadata.get("type", "unknown")
        page = doc.metadata.get("page", "?")
        if doc_type == "text":
            preview = doc.page_content[:100] + "..." if len(doc.page_content) > 100 else doc.page_content
            print(f"  - Text from page {page}: {preview}")
        else:
            print(f"  - Image from page {page}")
    print("\n")
    
    return response.content

In [25]:
if __name__ == "__main__":
    # Example queries
    queries = [
        "Summarize the main findings from the document",
        "What visual elements are present in the document?"
    ]
    
    for query in queries:
        print(f"\nQuery: {query}")
        print("-" * 50)
        answer = multimodal_pdf_rag_pipeline(query)
        print(f"Answer: {answer}")
        print("=" * 70)


Query: Summarize the main findings from the document
--------------------------------------------------

--- [Local Retrieval: Found 3 chunks] ---
Answer:  The main findings from the document are that Flamingo, a pretrained model, outperforms other methods on various tasks when fine-tuning on all nine tasks where it does not achieve state-of-the-art (SotA) with few-shot learning. It sets new SotA on five of these tasks. The document also provides an ablation study comparing Flamingo to SotA when fine-tuning Flamingo, and the results show that Flamingo outperforms other methods in most cases. Additionally, the document discusses the impact of different parameters and steps on the performance of Flamingo. 

Query: What visual elements are present in the document?
--------------------------------------------------

--- [Local Retrieval: Found 3 chunks] ---
Answer:  The document contains mathematical equations and text related to visual question answering (VQA) tasks, as well as various d